In [152]:
%matplotlib inline 
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
import os
import numpy as np
import pandas as pd
import re

We create a list for all the features we want in our dataframe

In [153]:
dates = []
prices = []
locations = []
pool = []
garage = []
type_col = [] #the type of property
bedrooms = []
sqm = []

In [154]:
#The text of announcements will be placed in this array
properties = []

Inspecting most of the file, it was concluded that the list of properties are defined by the classes named 'classified_list' and 'ca-Classifieds_PostListItem'.

The first four letters of the name of the files correspond to the date.

In [155]:
# iterate over files in
# that directory
for filename in os.listdir('./data'):
    
    #The first 4 characters correspond to the year
    date = filename[0:4]

    page = BeautifulSoup(open("data/" + filename,encoding="utf8"), "html.parser")
    for ultag in page.find_all('ul', {'class': ['classified_list','ca-Classifieds_PostList']}):
        for litag in ultag.find_all('li'):
            properties.append(litag.text)
            dates.append(date)

In [156]:
for p in properties:
    
    #find all the prices
    price = re.findall(r'([\€][,\d]+)', p);
    if not price:
        prices.append(np.nan);
    else:
        prices.append(price)
        
        
    #find the type of property
    type_of_prop=["house","apartment","penthouse","villa","maisonette","Apartment","House","Penthouse","Villa","Maisonette"]
    flag = False
    for kind in type_of_prop:
        if kind in p:
            #print(kind)
            type_col.append(kind)
            flag = True
            break;
        else: 
            flag = False               
    if (flag==False):
        type_col.append(np.nan)
        
        
    
    #find the location
    pattern = '(?<=\n)(.*?)(?=\.)'
    loc = re.findall(pattern,p)    
    if  ("PROPERTIES" in p or "FOCUSED" in p or "REDUCTIONS" in p or "RESTAURANT" in p):
        locations.append(np.nan)
    else:
        locations.append(loc)
    
    
    #find pools
    if "pool" in p or "Pool" in p:
        pool.append('Yes')
    else:
        pool.append('No')
        
    #find garage
    if "garage" in p or "Garage" in p:
        garage.append('Yes')
    else:
        garage.append('No')
    
    #find the number of bedrooms
    if 'bedroom' in p: 
        result = re.search("(\d|one|two|three|four|five)+\s(?:double.)*(bedrooms?)",p)
        if result is None:
            bedrooms.append(np.nan)
        else:    
            bedrooms.append(result.group(1))
    else:
        bedrooms.append(np.nan)
    
    #sqm
    if 'sq' in p or 'sqm' in p or 'sq.m' in p: 
        result = re.search("(\d)+\s*(sq\.*m?)",p)
        if result is None:
            sqm.append(np.nan)
        else:    
            sqm.append(result.group())
    else:
        sqm.append(np.nan)
        


In [157]:
print("number of properties: ",len(properties))

print(len(type_col))
print(len(prices))
print(len(pool))
print(len(garage))
print(len(locations))
print(len(dates))
print(len(bedrooms))
print(len(sqm))


number of properties:  138915
138915
138915
138915
138915
138915
138915
138915
138915


Now the data needs to be cleaned.

Clean prices

In [158]:
#Since the numbers are written using the comma, the latter will be deleted and type of the element will converted to integer
for i in range (len(prices)):
    
    #try-except to ignore nan errors
    
    try:
        prices[i] = prices[i][0]  #converting lists to string          
        prices[i] = prices[i].replace(',','') #removing comma
        prices[i] = prices[i][1:] #removing euro symbol
        prices[i] = int(prices[i]) #converting string to integer
    except:
        continue
    
    if prices[i] < 10:
        prices[i] = prices[i] * 1000000
    
    elif prices[i] >= 10 and prices[i] <= 10000:
        prices[i] = np.nan
    

Clean locations

Fist we change each element of location from list type to string

Then we split the string after a delimiter is encountered. This means that GOZO will be considered a unique location.

We clean by putting string greater than 20 characters as missing values

In [159]:
delimiters =[",","/","(",":"]

for i in range (len(locations)):
    
    try:
        locations[i]=locations[i][0]
    except:
        continue
        
    try:
        locations[i]=locations[i].upper()
    except:
        continue    
      
    try:
        for d in delimiters:
            if d in locations[i]:
                locations[i]=locations[i].split(d,1)[0]
    except:
        continue
    
    try:
        if(locations[i][0].isalpha() == False):
            locations[i]=np.nan
    except:
        continue
    
    try:
        if len(locations[i]) >= 20:
            locations[i] = np.nan
    except:
        continue
  
        
        

In [160]:
locations

[nan,
 'BALZAN',
 'GOZO',
 'GOZO',
 'GOZO',
 'MARSASCALA',
 'QAWRA',
 'QAWRA',
 'SIĠĠIEWI',
 "ST PAUL' S BAY",
 'ST VENERA',
 'ST VENERA',
 "TA' XBIEX",
 'TARXIEN',
 'THE VILLAGE',
 'VICTORIA GARDENS',
 'ŻEBBUĠ',
 nan,
 'ATTARD',
 'BALZAN',
 'BALZAN',
 'BIRKIRKARA',
 'BIRKIRKARA',
 'COSPICUA ',
 'FLEUR-DE-LYS',
 'GOZO',
 'GOZO',
 'GOZO',
 'GOZO',
 'KAPPARA',
 'MARSASCALA',
 'MARSASCALA',
 'MĠARR',
 'MTARFA',
 'NAXXAR',
 'SAN PAWL TAT-TARĠA',
 'SANTA LUĊIJA',
 'SIĠĠIEWI',
 'SLIEMA',
 'SLIEMA',
 "ST PAUL' S BAY",
 'ST VENERA',
 'ST VENERA',
 "TA' XBIEX",
 'THE VILLAGE',
 'VALLETTA',
 'VALLETTA',
 'VALLETTA',
 'VALLETTA',
 'ŻABBAR',
 'ŻEBBUĠ',
 'ŻEBBUĠ',
 nan,
 'BALZAN',
 'GOZO',
 'GOZO',
 'GOZO',
 'MARSASCALA',
 'QAWRA',
 "ST PAUL' S BAY",
 'ST VENERA',
 'ST VENERA',
 'TARXIEN',
 nan,
 'GOZO',
 'GOZO',
 'GOZO',
 'NAXXAR',
 'SLIEMA',
 [],
 nan,
 nan,
 nan,
 nan,
 nan,
 'APARTMENTS',
 'ATTARD',
 'ATTARD',
 'ATTARD',
 'ATTARD',
 'ATTARD',
 'ATTARD',
 'ATTARD',
 'ATTARD',
 'ATTARD',
 'ATTARD

Clean type_col by writing all the names in lowercase.

In [161]:
for i in range (len(type_col)):
    try:
        type_col[i]=type_col[i].lower()
    except:
        continue

type_col

[nan,
 'apartment',
 'house',
 'house',
 'maisonette',
 'apartment',
 'apartment',
 'house',
 'maisonette',
 'apartment',
 'house',
 nan,
 'villa',
 'house',
 'maisonette',
 nan,
 'house',
 nan,
 'villa',
 'apartment',
 nan,
 'house',
 'house',
 'house',
 'house',
 'house',
 'apartment',
 'house',
 'house',
 'villa',
 'apartment',
 'apartment',
 'maisonette',
 'house',
 'house',
 nan,
 'house',
 'maisonette',
 'apartment',
 'house',
 'house',
 'maisonette',
 'apartment',
 'apartment',
 'villa',
 'apartment',
 'house',
 'house',
 nan,
 'maisonette',
 'house',
 'house',
 nan,
 'apartment',
 'apartment',
 'house',
 'house',
 'apartment',
 'house',
 'apartment',
 'house',
 'maisonette',
 'house',
 nan,
 'apartment',
 'apartment',
 nan,
 'house',
 'apartment',
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 'maisonette',
 'apartment',
 'apartment',
 'maisonette',
 'maisonette',
 'apartment',
 nan,
 'house',
 'apartment',
 'house',
 'house',
 'maisonette',
 'maisonette',
 'apartment',
 'house',
 'house

Clean number of bedrooms

In [162]:
help_dict = {
    'one': '1',
    'two': '2',
    'three': '3',
    'four': '4',
    'five': '5',
    'six': '6',
    'seven': '7',
    'eight': '8',
    'nine': '9',
    'zero': '0'
}

for i in range(len(bedrooms)):
    try:
        bedrooms[i] = ''.join(help_dict[ele] for ele in bedrooms[i].split())
    except:
        continue

Clean sqm

In [163]:
for i in range(len(sqm)):
    try:
        sqm[i]=re.sub("[^0-9]", "", sqm[i])
        sqm[i] = int(sqm[i]) #converting string to integer
        
        if sqm[i] < 20:
            sqm[i] = np.nan
    except:
        continue; 

In [164]:
type(sqm[16])

int

In [165]:
df = pd.DataFrame(list(zip(locations,prices,type_col,pool,garage,dates,bedrooms,sqm)),
                 columns=['LOCATION','PRICE','TYPE_OF_PROPERTY','POOL','GARAGE','DATE','BEDROOM','SQM'])

In [166]:
df

,LOCATION,PRICE,TYPE_OF_PROPERTY,POOL,GARAGE,DATE,BEDROOM,SQM
0,NaN,NaN,NaN,No,No,2015,NaN,NaN
1,BALZAN,152000.0,apartment,No,No,2015,3,NaN
2,GOZO,185000.0,house,No,No,2015,NaN,NaN
3,GOZO,293000.0,house,Yes,No,2015,3,NaN
4,GOZO,145000.0,maisonette,No,No,2015,NaN,NaN
...,...,...,...,...,...,...,...,...
138910,NaN,NaN,NaN,No,No,2022,NaN,NaN
138911,NaN,NaN,NaN,No,No,2022,NaN,NaN
138912,NaN,NaN,NaN,No,No,2022,NaN,NaN
138913,ST JULIANS,830000.0,NaN,No,No,2022,NaN,300.0


In [167]:
#check missing values
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 138915 entries, 0 to 138914
Data columns (total 8 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   LOCATION          135341 non-null  object 
 1   PRICE             119233 non-null  float64
 2   TYPE_OF_PROPERTY  123235 non-null  object 
 3   POOL              138915 non-null  object 
 4   GARAGE            138915 non-null  object 
 5   DATE              138915 non-null  object 
 6   BEDROOM           69056 non-null   object 
 7   SQM               27293 non-null   float64
dtypes: float64(2), object(6)
memory usage: 8.5+ MB


In [168]:
df.to_csv('raw_data.csv')

Dropping all the missing values makes us lose too much data

In [169]:
no_na=df.dropna()

In [170]:
no_na

,LOCATION,PRICE,TYPE_OF_PROPERTY,POOL,GARAGE,DATE,BEDROOM,SQM
16,ŻEBBUĠ,950000.0,house,Yes,No,2015,3,200.0
38,SLIEMA,220000.0,apartment,No,No,2015,3,175.0
43,TA' XBIEX,850000.0,apartment,No,Yes,2015,3,270.0
50,ŻEBBUĠ,950000.0,house,Yes,No,2015,3,200.0
98,BAĦAR IĊ-ĊAGĦAQ,650000.0,villa,Yes,Yes,2015,3,360.0
...,...,...,...,...,...,...,...,...
138865,QAWRA,295000.0,maisonette,No,No,2022,2,20.0
138869,QORMI,234000.0,house,No,No,2022,4,126.0
138870,QORMI,350000.0,maisonette,No,Yes,2022,3,150.0
138880,SLIEMA,350000.0,apartment,No,No,2022,2,115.0


HANDLING MISSING VALUES 

First all the rows with price as missing value will be removed since they most likely do not correspond to valid announcement

In [171]:
df = df[df['PRICE'].notna()]
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 119233 entries, 1 to 138913
Data columns (total 8 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   LOCATION          117853 non-null  object 
 1   PRICE             119233 non-null  float64
 2   TYPE_OF_PROPERTY  108683 non-null  object 
 3   POOL              119233 non-null  object 
 4   GARAGE            119233 non-null  object 
 5   DATE              119233 non-null  object 
 6   BEDROOM           61846 non-null   object 
 7   SQM               23140 non-null   float64
dtypes: float64(2), object(6)
memory usage: 8.2+ MB


The same can be done with TYPE_OF_PROPERTY and LOCATION because the missing values are a small percentage

In [172]:
df = df[df['TYPE_OF_PROPERTY'].notna()]
df = df[df['LOCATION'].notna()]
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 107549 entries, 1 to 138909
Data columns (total 8 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   LOCATION          107549 non-null  object 
 1   PRICE             107549 non-null  float64
 2   TYPE_OF_PROPERTY  107549 non-null  object 
 3   POOL              107549 non-null  object 
 4   GARAGE            107549 non-null  object 
 5   DATE              107549 non-null  object 
 6   BEDROOM           58424 non-null   object 
 7   SQM               20230 non-null   float64
dtypes: float64(2), object(6)
memory usage: 7.4+ MB


We observe an high percentage of missing value in the column BEDROOM. We replace the missing values with the most common value 

In [173]:
df['BEDROOM'].value_counts()

3    37267
2    14151
4     4303
1     2199
5      493
0        9
7        2
Name: BEDROOM, dtype: int64

In [174]:
mean_b = df['BEDROOM'].mode()
mean_b

0    3
dtype: object

Replacing missing values with 3

In [175]:
df['BEDROOM']=df['BEDROOM'].fillna(3)

In [176]:
df

,LOCATION,PRICE,TYPE_OF_PROPERTY,POOL,GARAGE,DATE,BEDROOM,SQM
1,BALZAN,152000.0,apartment,No,No,2015,3,NaN
2,GOZO,185000.0,house,No,No,2015,3,NaN
3,GOZO,293000.0,house,Yes,No,2015,3,NaN
4,GOZO,145000.0,maisonette,No,No,2015,3,NaN
5,MARSASCALA,85000.0,apartment,No,No,2015,2,NaN
...,...,...,...,...,...,...,...,...
138903,SWIEQI,375000.0,maisonette,No,Yes,2022,3,NaN
138905,TARXIEN,310000.0,maisonette,No,No,2022,3,NaN
138906,VALLETTA,575000.0,house,No,No,2022,3,NaN
138908,ŻEBBUĠ,245000.0,apartment,No,No,2022,3,NaN


There is a high percentage of missing values for SQM. Intuitively the column should be dropped, but we'll keep it because it could be useful for some statistical analysis 

In [177]:
df.reset_index(drop=True)

,LOCATION,PRICE,TYPE_OF_PROPERTY,POOL,GARAGE,DATE,BEDROOM,SQM
0,BALZAN,152000.0,apartment,No,No,2015,3,NaN
1,GOZO,185000.0,house,No,No,2015,3,NaN
2,GOZO,293000.0,house,Yes,No,2015,3,NaN
3,GOZO,145000.0,maisonette,No,No,2015,3,NaN
4,MARSASCALA,85000.0,apartment,No,No,2015,2,NaN
...,...,...,...,...,...,...,...,...
107544,SWIEQI,375000.0,maisonette,No,Yes,2022,3,NaN
107545,TARXIEN,310000.0,maisonette,No,No,2022,3,NaN
107546,VALLETTA,575000.0,house,No,No,2022,3,NaN
107547,ŻEBBUĠ,245000.0,apartment,No,No,2022,3,NaN


Cleaning locations is more troublesome than excepected. Looking at the unique values we can create a set of locations which will be in our dataset.

In [178]:
df['LOCATION'].unique()

array(['BALZAN', 'GOZO', 'MARSASCALA', 'QAWRA', 'SIĠĠIEWI',
       "ST PAUL' S BAY", 'ST VENERA', "TA' XBIEX", 'TARXIEN',
       'THE VILLAGE', 'ŻEBBUĠ', 'ATTARD', 'BIRKIRKARA', 'COSPICUA ',
       'KAPPARA', 'NAXXAR', 'SANTA LUĊIJA', 'SLIEMA', 'VALLETTA',
       'ŻABBAR', 'BAĦAR IĊ-ĊAGĦAQ', 'BAHAR IĊ-ĊAGĦAQ', 'BAĦRIJA',
       'BALZAN VALLEY', 'BIRGUMA', 'BIRŻEBBUĠA', 'BUĠIBBA', 'DINGLI',
       'FGURA', 'FLEUR-DE-LYS', 'FORT CAMBRIDGE', 'GĦARGĦUR', 'GĦAXAQ',
       'GUARDAMANGIA', 'GUDJA', 'GŻIRA', 'ĦAMRUN', 'IKLIN', 'KALKARA',
       'KIRKOP', 'LIJA', 'LUQA', 'MADLIENA', 'MANIKATA', 'MARSA',
       'MARSAXLOKK', 'MELLIEĦA', 'MĠARR', 'MOSTA', 'MQABBA',
       'MSIDA CIRCUS', 'MSIDA', 'PAOLA', 'PIETÀ', 'PORTOMASO', 'PWALES',
       'QORMI', 'RABAT', 'SALINA', 'SAN ĠWANN', 'SAN PAWL TAT-TARĠA',
       'SENGLEA ', 'SLIEMA ', 'ST JULIANS', 'SWATAR', 'SWIEQI',
       "TA' GIORNI", "TA' XBIEX ", 'TAL-IBRAĠ', 'THE STRAND',
       'VITTORIOSA ', 'XEMXIJA BAY', 'XEMXIJA', 'ŻEBBIEGĦ', 'ŻEJTUN'

In [179]:
df['LOCATION'].value_counts().head(80)

SLIEMA            7505
GOZO              7381
MOSTA             5239
MARSASCALA        4483
ST JULIANS        4469
                  ... 
SAFI               148
VITTORIOSA         135
SALINA             133
ST PAUL' S         127
FORT CAMBRIDGE     114
Name: LOCATION, Length: 80, dtype: int64

In [180]:
#top 80 location
top_80=df['LOCATION'].value_counts().head(80).index.tolist()

In [181]:
top_80

['SLIEMA',
 'GOZO',
 'MOSTA',
 'MARSASCALA',
 'ST JULIANS',
 'QAWRA',
 'BIRKIRKARA',
 'ATTARD',
 'MELLIEĦA',
 'NAXXAR',
 'GŻIRA',
 'MSIDA',
 'SWIEQI',
 'ŻEJTUN',
 'FGURA',
 'GĦARGĦUR',
 'ŻEBBUĠ',
 'ŻABBAR',
 "ST PAUL' S BAY",
 'ST VENERA',
 'TAL-IBRAĠ',
 'SAN ĠWANN',
 'BALZAN',
 'ŻURRIEQ',
 'RABAT',
 'SIĠĠIEWI',
 'QORMI',
 'TARXIEN',
 'BUĠIBBA',
 'LIJA',
 'VALLETTA',
 'PAOLA',
 'SAN PAWL TAT-TARĠA',
 'ĦAMRUN',
 'BIRŻEBBUĠA',
 'BAĦAR IĊ-ĊAGĦAQ',
 'KAPPARA',
 'PIETÀ',
 'XEMXIJA',
 'SWATAR',
 'MĠARR',
 'GĦAXAQ',
 'MADLIENA',
 'IKLIN',
 'KALKARA',
 'LUQA',
 'MARSAXLOKK',
 'ST PAUL’S BAY',
 'MANIKATA',
 "TA' XBIEX",
 'GUARDAMANGIA',
 'SENGLEA ',
 'GUDJA',
 'QRENDI',
 'XGĦAJRA',
 'COSPICUA ',
 'PORTOMASO',
 'SLIEMA ',
 'DINGLI',
 'MQABBA',
 'FLORIANA',
 'PEMBROKE',
 'ŻEBBIEGĦ',
 'KIRKOP',
 'BAĦRIJA',
 'VITTORIOSA ',
 'BIRGUMA',
 'COSPICUA',
 'TIGNÉ POINT',
 'THE VILLAGE',
 'MENSIJA',
 'SENGLEA',
 'MTARFA',
 'MARSA',
 'BURMARRAD',
 'SAFI',
 'VITTORIOSA',
 'SALINA',
 "ST PAUL' S",
 'FORT CAMBR

In [182]:
top_80.remove('THE VILLAGE')

In [183]:
data_top_loc=df[df['LOCATION'].isin(top_80)].reset_index(drop=True)

In [184]:
data_top_loc['LOCATION'].unique()

array(['BALZAN', 'GOZO', 'MARSASCALA', 'QAWRA', 'SIĠĠIEWI',
       "ST PAUL' S BAY", 'ST VENERA', "TA' XBIEX", 'TARXIEN', 'ŻEBBUĠ',
       'ATTARD', 'BIRKIRKARA', 'COSPICUA ', 'KAPPARA', 'NAXXAR', 'SLIEMA',
       'VALLETTA', 'ŻABBAR', 'BAĦAR IĊ-ĊAGĦAQ', 'BAĦRIJA', 'BIRGUMA',
       'BIRŻEBBUĠA', 'BUĠIBBA', 'DINGLI', 'FGURA', 'FORT CAMBRIDGE',
       'GĦARGĦUR', 'GĦAXAQ', 'GUARDAMANGIA', 'GUDJA', 'GŻIRA', 'ĦAMRUN',
       'IKLIN', 'KALKARA', 'KIRKOP', 'LIJA', 'LUQA', 'MADLIENA',
       'MANIKATA', 'MARSA', 'MARSAXLOKK', 'MELLIEĦA', 'MĠARR', 'MOSTA',
       'MQABBA', 'MSIDA', 'PAOLA', 'PIETÀ', 'PORTOMASO', 'QORMI', 'RABAT',
       'SALINA', 'SAN ĠWANN', 'SAN PAWL TAT-TARĠA', 'SENGLEA ', 'SLIEMA ',
       'ST JULIANS', 'SWATAR', 'SWIEQI', 'TAL-IBRAĠ', 'VITTORIOSA ',
       'XEMXIJA', 'ŻEBBIEGĦ', 'ŻEJTUN', 'ŻURRIEQ', 'COSPICUA', 'FLORIANA',
       'MTARFA', 'QRENDI', 'SAFI', 'VITTORIOSA', 'BURMARRAD', 'PEMBROKE',
       "ST PAUL' S", 'XGĦAJRA', 'TIGNÉ POINT', 'MENSIJA', 'SENGLEA',
       

In [185]:
data_top_loc['LOCATION'].replace('COSPICUA ','COSPICUA',inplace=True)
data_top_loc['LOCATION'].replace('VITTORIOSA ','VITTORIOSA',inplace=True)
data_top_loc['LOCATION'].replace('ST PAUL’S BAY','ST PAUL',inplace=True)
data_top_loc['LOCATION'].replace("ST PAUL' S",'ST PAUL',inplace=True)
data_top_loc['LOCATION'].replace("SAN PAWL TAT-TARĠA",'ST PAUL',inplace=True)
data_top_loc['LOCATION'].replace("ST PAUL' S BAY",'ST PAUL',inplace=True)

In [186]:
data_top_loc.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103853 entries, 0 to 103852
Data columns (total 8 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   LOCATION          103853 non-null  object 
 1   PRICE             103853 non-null  float64
 2   TYPE_OF_PROPERTY  103853 non-null  object 
 3   POOL              103853 non-null  object 
 4   GARAGE            103853 non-null  object 
 5   DATE              103853 non-null  object 
 6   BEDROOM           103853 non-null  object 
 7   SQM               19336 non-null   float64
dtypes: float64(2), object(6)
memory usage: 6.3+ MB


In [187]:
data_top_loc

,LOCATION,PRICE,TYPE_OF_PROPERTY,POOL,GARAGE,DATE,BEDROOM,SQM
0,BALZAN,152000.0,apartment,No,No,2015,3,NaN
1,GOZO,185000.0,house,No,No,2015,3,NaN
2,GOZO,293000.0,house,Yes,No,2015,3,NaN
3,GOZO,145000.0,maisonette,No,No,2015,3,NaN
4,MARSASCALA,85000.0,apartment,No,No,2015,2,NaN
...,...,...,...,...,...,...,...,...
103848,SWIEQI,375000.0,maisonette,No,Yes,2022,3,NaN
103849,TARXIEN,310000.0,maisonette,No,No,2022,3,NaN
103850,VALLETTA,575000.0,house,No,No,2022,3,NaN
103851,ŻEBBUĠ,245000.0,apartment,No,No,2022,3,NaN


In [188]:
data_top_loc.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103853 entries, 0 to 103852
Data columns (total 8 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   LOCATION          103853 non-null  object 
 1   PRICE             103853 non-null  float64
 2   TYPE_OF_PROPERTY  103853 non-null  object 
 3   POOL              103853 non-null  object 
 4   GARAGE            103853 non-null  object 
 5   DATE              103853 non-null  object 
 6   BEDROOM           103853 non-null  object 
 7   SQM               19336 non-null   float64
dtypes: float64(2), object(6)
memory usage: 6.3+ MB


In [189]:
data_top_loc.to_csv('malta_properties.csv')